In [ ]:


export=False


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'BEA'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / ' BEA'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'

with path_func.open("r") as f:
    exec(f.read())



Inflation-adjusted

In [ ]:


file_in = path_raw / 'CAGDP9' / 'CAGDP9_MSA_2001_2023.csv'
df_rdp = pd.read_csv(file_in, na_values="(D)")

df_rdp.columns = df_rdp.columns.str.strip()


# Filtering and selecting columns
df_rdp = df_rdp[df_rdp['LineCode'] < 87]
df_rdp = df_rdp.dropna(subset=['GeoName'])
df_rdp = df_rdp[df_rdp['Description'] != 'Addenda:']

# Replacing parts of GeoName
df_rdp['GeoFIPS'] = df_rdp['GeoFIPS'].str.replace('"', '')
df_rdp['GeoName'] = df_rdp['GeoName'].str.replace(r"\(.*", "", regex=True)
df_rdp['GeoName'] = df_rdp['GeoName'].str.strip()
df_rdp['Description'] = df_rdp['Description'].str.strip()

msa_peers = [
    'Atlanta, GA'
    , 'Columbus, OH'
    , 'Kansas City, MO'
    , 'Minneapolis, MN'
    , 'Portland, OR'
    , 'Seattle, WA'
    , 'Sacramento, CA'
    , 'Yuba City, CA'
]


df_rdp['GeoName'] = df_rdp['GeoName'].map(peer_msa_labels)
df_rdp = df_rdp[df_rdp['GeoName'].isin(msa_peers)]
df_rdp = df_rdp.drop(['Region', 'TableName', 'LineCode', 'Unit'], axis=1)


df_rdp = df_rdp.rename(columns={'GeoFIPS':'MSA ID', 'GeoName':'MSA', 'IndustryClassification':'NAICS Code', 'Description':'Industry'})

df_rdp = pd.melt(df_rdp, id_vars=['MSA ID', 'MSA', 'NAICS Code', 'Industry'], var_name='Year', value_name='GRP (Thousands of Dollars)')
df_rdp['Year'] = df_rdp['Year'].astype(int)
df_rdp['MSA ID'] = df_rdp['MSA ID'].astype(int)

df_rdp = df_rdp[df_rdp['Industry'] == 'All industry total']
df_rdp = df_rdp.drop(['NAICS Code', 'Industry'], axis=1)


df_rdp.loc[df_rdp['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
df_rdp['MSA ID'] = df_rdp['MSA ID'].astype(str)
df_rdp.loc[df_rdp['MSA'] == 'SACOG', 'MSA ID'] = '40900, 49700'
df_rdp = df_rdp.groupby(['MSA ID', 'MSA', 'Year'], as_index=False)['GRP (Thousands of Dollars)'].sum()



df_rdp = df_rdp.sort_values(['MSA ID', 'Year'], ascending = [True, True]).reset_index(drop=True)


df_rdp['AnnualPctDiff'] = df_rdp['GRP (Thousands of Dollars)'].pct_change()
df_rdp.loc[df_rdp['Year'] == 2001, 'AnnualPctDiff'] = np.nan
df_rdp.loc[df_rdp['AnnualPctDiff'] == np.inf, 'AnnualPctDiff'] = np.nan

year_max = np.max(df_rdp['Year'].unique())
year_min = np.min(df_rdp['Year'].unique())


col_pct_change = f'PctChange_{year_min}_{year_max}'
col_annual_pct_change = f'AnnualPctDiff_{year_min}_{year_max}'


df_rdp2 = df_rdp.copy()
df_rdp2 = df_rdp2[(df_rdp2['Year'] == year_max) | (df_rdp2['Year'] == year_min)]
df_rdp2 = df_rdp2.drop('AnnualPctDiff', axis = 1)
df_rdp2[col_pct_change] = df_rdp2['GRP (Thousands of Dollars)'].pct_change()
df_rdp2.loc[df_rdp2['Year'] == year_min, col_pct_change] = np.nan
df_rdp2.loc[df_rdp2[col_pct_change] == np.inf, col_pct_change] = np.nan
df_rdp2 = df_rdp2[df_rdp2['Year'] == year_max]
df_rdp2 = df_rdp2.drop(['Year', 'GRP (Thousands of Dollars)'], axis=1)

df_rdp3 = df_rdp.copy()
df_rdp3 = df_rdp3.groupby(['MSA'], as_index = False).agg(weighted_mean_grp = ('AnnualPctDiff', 'mean'))
df_rdp3 = df_rdp3.rename(columns={'weighted_mean_grp':col_annual_pct_change})


df_rdp2 = df_rdp2.merge(df_rdp3, on = 'MSA')

df_rdp   = df_rdp.sort_values(['MSA', 'Year'], ascending=[True, False])
df_rdp2 = df_rdp2.sort_values(['MSA'        ], ascending=[True       ])



df_rdp  = df_rdp .reset_index(drop=True)
df_rdp2 = df_rdp2.reset_index(drop=True)

display(df_rdp.head())
display(df_rdp2.head())




fig = px.line(df_rdp, x = 'Year', y = 'GRP (Thousands of Dollars)', color = 'MSA', markers=True)
fig.update_xaxes(tick0=0, dtick=1, range=[2000.5, 2023.5])
fig.update_yaxes(tick0=0, tickprefix='$', tickformat = ',.0f')
fig.update_traces(hovertemplate='%{y}')


fig.show()




# fig = px.line(df_rdp, x = 'Year', y = 'AnnualPctDiff', color = 'MSA', markers=True)
# fig.update_xaxes(tick0=0, dtick=1, range=[2000.5, 2023.5])
# fig.update_traces(hovertemplate='%{y}')


# fig.show()




if export:

    sample_type = 'BEA'
    indicator = 'Output_1'
    year_start = int(year_min)
    year_end = int(year_max)
    geography = 'MSA'

    # Create about documentation page for export
    df_about = write_about(sample_type, indicator, year_start, year_end, path_config0)

    print("Visual representation of the output for:", indicator)
    display(df_about)

    file_out = path_csm / f'{indicator} {geography} {sample_type}_ChamberStudy2026.xlsx'
    with pd.ExcelWriter(file_out, engine='openpyxl') as writer:
        df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
        df_rdp  .to_excel(writer, index = False, sheet_name = 'MSA'                  )
        df_rdp2 .to_excel(writer, index = False, sheet_name = 'Rates'                )

    print()
    print("Successfully exported!")

    





**************

Not adjusted for inflation

In [ ]:


file_in = path_raw / 'CAGDP2' / 'CAGDP2_MSA_2001_2023.csv'
df_rdp = pd.read_csv(file_in, na_values="(D)")

df_rdp.columns = df_rdp.columns.str.strip()


# Filtering and selecting columns
df_rdp = df_rdp[df_rdp['LineCode'] < 87]
df_rdp = df_rdp.dropna(subset=['GeoName'])
df_rdp = df_rdp[df_rdp['Description'] != 'Addenda:']

# Replacing parts of GeoName
df_rdp['GeoFIPS'] = df_rdp['GeoFIPS'].str.replace('"', '')
df_rdp['GeoName'] = df_rdp['GeoName'].str.replace(r"\(.*", "", regex=True)
df_rdp['GeoName'] = df_rdp['GeoName'].str.strip()
df_rdp['Description'] = df_rdp['Description'].str.strip()

msa_peers = [
    'Atlanta, GA'
    , 'Columbus, OH'
    , 'Kansas City, MO'
    , 'Minneapolis, MN'
    , 'Portland, OR'
    , 'Seattle, WA'
    , 'Sacramento, CA'
    , 'Yuba City, CA'
]


df_rdp['GeoName'] = df_rdp['GeoName'].map(peer_msa_labels)
df_rdp = df_rdp[df_rdp['GeoName'].isin(msa_peers)]
df_rdp = df_rdp.drop(['Region', 'TableName', 'LineCode', 'Unit'], axis=1)


df_rdp = df_rdp.rename(columns={'GeoFIPS':'MSA ID', 'GeoName':'MSA', 'IndustryClassification':'NAICS Code', 'Description':'Industry'})

df_rdp = pd.melt(df_rdp, id_vars=['MSA ID', 'MSA', 'NAICS Code', 'Industry'], var_name='Year', value_name='GRP (Thousands of Dollars)')
df_rdp['Year'] = df_rdp['Year'].astype(int)
df_rdp['MSA ID'] = df_rdp['MSA ID'].astype(int)

df_rdp = df_rdp[df_rdp['Industry'] == 'All industry total']
df_rdp = df_rdp.drop(['NAICS Code', 'Industry'], axis=1)


df_rdp.loc[df_rdp['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
df_rdp['MSA ID'] = df_rdp['MSA ID'].astype(str)
df_rdp.loc[df_rdp['MSA'] == 'SACOG', 'MSA ID'] = '40900, 49700'
df_rdp = df_rdp.groupby(['MSA ID', 'MSA', 'Year'], as_index=False)['GRP (Thousands of Dollars)'].sum()



df_rdp = df_rdp.sort_values(['MSA ID', 'Year'], ascending = [True, True]).reset_index(drop=True)


df_rdp['AnnualPctDiff'] = df_rdp['GRP (Thousands of Dollars)'].pct_change()
df_rdp.loc[df_rdp['Year'] == 2001, 'AnnualPctDiff'] = np.nan
df_rdp.loc[df_rdp['AnnualPctDiff'] == np.inf, 'AnnualPctDiff'] = np.nan

year_max = np.max(df_rdp['Year'].unique())
year_min = np.min(df_rdp['Year'].unique())


col_pct_change = f'PctChange_{year_min}_{year_max}'
col_annual_pct_change = f'AnnualPctDiff_{year_min}_{year_max}'


df_rdp2 = df_rdp.copy()
df_rdp2 = df_rdp2[(df_rdp2['Year'] == year_max) | (df_rdp2['Year'] == year_min)]
df_rdp2 = df_rdp2.drop('AnnualPctDiff', axis = 1)
df_rdp2[col_pct_change] = df_rdp2['GRP (Thousands of Dollars)'].pct_change()
df_rdp2.loc[df_rdp2['Year'] == year_min, col_pct_change] = np.nan
df_rdp2.loc[df_rdp2[col_pct_change] == np.inf, col_pct_change] = np.nan
df_rdp2 = df_rdp2[df_rdp2['Year'] == year_max]
df_rdp2 = df_rdp2.drop(['Year', 'GRP (Thousands of Dollars)'], axis=1)

df_rdp3 = df_rdp.copy()
df_rdp3 = df_rdp3.groupby(['MSA'], as_index = False).agg(weighted_mean_grp = ('AnnualPctDiff', 'mean'))
df_rdp3 = df_rdp3.rename(columns={'weighted_mean_grp':col_annual_pct_change})


df_rdp2 = df_rdp2.merge(df_rdp3, on = 'MSA')

df_rdp   = df_rdp.sort_values(['MSA', 'Year'], ascending=[True, False])
df_rdp2 = df_rdp2.sort_values(['MSA'        ], ascending=[True       ])



df_rdp  = df_rdp .reset_index(drop=True)
df_rdp2 = df_rdp2.reset_index(drop=True)

display(df_rdp.head())
display(df_rdp2.head())




fig2 = px.line(df_rdp, x = 'Year', y = 'GRP (Thousands of Dollars)', color = 'MSA', markers=True)
fig2.update_xaxes(tick0=0, dtick=1, range=[2000.5, 2023.5])
fig2.update_yaxes(tick0=0, tickprefix='$', tickformat = ',.0f')
fig2.update_traces(hovertemplate='%{y}')


fig2.show()




In [ ]:
fig.show()